# 03 — Sentinel-2 representative-year comparison
**Purpose:** inspect RGB, NDSI, snow masks, and valid observations for representative post-monsoon years.  
**Inputs:** GLIMS ROI and Phase 3 composites.  
**Outputs:** interactive comparison map; optional HTML export.  
**Runtime:** usually a few minutes because imagery renders on demand.  
**Dependencies:** authenticated Earth Engine session and Phase 1 configuration.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

%cd "/content/drive/MyDrive/langtang-glacier-ai"

Mounted at /content/drive
/content/drive/MyDrive/langtang-glacier-ai


In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import ee
from src.config import SETTINGS
from src.gee_utils import build_rois, initialize_earth_engine
from src.glacier_features import make_seasonal_composite
from src.utils import ensure_output_directories
from src.visualization import NDSI_VIS, RGB_VIS, make_study_area_map
ensure_output_directories()
EE_PROJECT = "my-youtube-api-keys-453716"  # Google Earth Engine Project ID

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    initialize_earth_engine(
        project=EE_PROJECT,
        authenticate=True,
    )


In [3]:
rois = build_rois()
visual_map = make_study_area_map(rois)
for year in SETTINGS.representative_years:
    image = make_seasonal_composite(rois['glacier_roi'], year)
    visual_map.addLayer(image, RGB_VIS, f'RGB {year}', year == SETTINGS.representative_years[-1])
    visual_map.addLayer(image.select('NDSI'), NDSI_VIS, f'NDSI {year}', False)
    visual_map.addLayer(image.select('snow').selfMask(), {'palette':['00D8FF']}, f'Snow proxy {year}', False)
    visual_map.addLayer(image.select('valid_observation_count'), {'min':0,'max':8,'palette':['black','yellow','green']}, f'Valid observations {year}', False)
visual_map.to_html(str(SETTINGS.output_maps / 'sentinel2_detailed_comparison.html'))
visual_map


Map(center=[28.251882136909924, 85.54672799144113], controls=(WidgetControl(options=['position', 'transparent_…

## Scientific reading guide
Compare only like seasons. Treat changes along shadowed slopes, cloud edges, and debris-covered tongue with skepticism. A changing snow proxy can reflect transient snow conditions and does not by itself demonstrate glacier-boundary retreat. Threshold sensitivity and reference imagery are required before publication.